# Dataset Preparation Pipeline & Label Validation
### AI-Based Smart Gate Monitoring System
**Author:** Siddharth Dhanush R (Team Lead)
**Objective:** Extract representative non-redundant frames from gate recordings, validate Darknet bounding-box format, partition dataset by recording session to prevent temporal data leakage, and generate `data.yaml`.

## 1. Setup Environment & Mount Storage

In [ ]:
# If using Google Colab, mount Google Drive to access authorized gate CCTV videos
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[INFO] Google Drive mounted successfully.")
except Exception:
    print("[INFO] Running in local/standalone environment.")

import os
import cv2
import glob
import yaml
import shutil
import numpy as np
import matplotlib.pyplot as plt

## 2. Frame Extraction with Fixed Stride Sampling
To avoid massive frame redundancy, we extract 1 frame per second (stride = 30 for 30fps video).

In [ ]:
def extract_frames(video_path, output_dir, stride=30, prefix="rec"):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    count = 0
    saved = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if count % stride == 0:
            filename = f"{prefix}_f{count:06d}.jpg"
            cv2.imwrite(os.path.join(output_dir, filename), frame)
            saved += 1
        count += 1
    cap.release()
    print(f"[INFO] Processed {video_path}: {saved} frames extracted from {count} total frames.")
    return saved

# Example usage:
# extract_frames("/content/drive/MyDrive/GateRecordings/GateCam_Morning.mp4", "ai/dataset/extracted_frames", stride=30, prefix="gate_morning")

## 3. Darknet YOLO Label Validation
Validates that all bounding boxes have valid normalized coordinates in [0.0, 1.0] and valid class IDs (0-5).

In [ ]:
def validate_labels(images_dir, labels_dir, num_classes=6):
    images = glob.glob(os.path.join(images_dir, "*.jpg"))
    valid_count = 0
    corrupted_count = 0
    class_counts = {i: 0 for i in range(num_classes)}
    
    for img_path in images:
        base = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(labels_dir, f"{base}.txt")
        if not os.path.exists(lbl_path):
            continue
        
        with open(lbl_path, "r") as f:
            lines = [l.strip() for l in f if l.strip()]
        
        valid = True
        for l in lines:
            parts = l.split()
            if len(parts) != 5:
                valid = False; break
            cls_id = int(parts[0])
            xc, yc, w, h = map(float, parts[1:])
            if cls_id < 0 or cls_id >= num_classes or not all(0.0 <= v <= 1.0 for v in [xc, yc, w, h]):
                valid = False; break
            class_counts[cls_id] += 1
        
        if valid:
            valid_count += 1
        else:
            corrupted_count += 1
            
    print(f"[VALIDATION] Valid Image-Label Pairs: {valid_count} | Corrupted/Invalid: {corrupted_count}")
    print("[CLASS DISTRIBUTION]", class_counts)
    return class_counts

# validate_labels("ai/dataset/images/train", "ai/dataset/labels/train", num_classes=6)

## 4. Partition Splits by Whole Recording Session
Prevents data leakage by ensuring entire recording sessions are kept distinct across train, val, and test.

In [ ]:
def create_leak_free_split(recording_dict, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    recs = list(recording_dict.keys())
    np.random.seed(42)
    np.random.shuffle(recs)
    
    n = len(recs)
    n_train = max(1, int(round(n * train_ratio)))
    n_val = max(1, int(round(n * val_ratio)))
    
    train_recs = recs[:n_train]
    val_recs = recs[n_train:n_train+n_val]
    test_recs = recs[n_train+n_val:]
    
    print(f"[SPLIT] Train recordings ({len(train_recs)}): {train_recs}")
    print(f"[SPLIT] Val recordings ({len(val_recs)}): {val_recs}")
    print(f"[SPLIT] Test recordings ({len(test_recs)}): {test_recs}")
    return train_recs, val_recs, test_recs

## 5. Generate `data.yaml`

In [ ]:
data_yaml_content = {
    'path': '/content/dataset',  # Path in Colab environment
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 6,
    'names': {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'bus', 5: 'truck'}
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)
print("[INFO] Created data.yaml for YOLOv8 training.")